# 07 — Normalization and Dimensionality Reduction

**Prerequisites:** Run [06_guide_assignment.ipynb](06_guide_assignment.ipynb) first. Requires `data/norman_assigned.h5ad`.

**What you'll learn:**
- Standard scRNA-seq normalization in the Perturb-seq context
- Why HVG selection and PCA should be anchored to NTC cells
- Cell cycle scoring and when to regress it out
- NTC-anchored PCA: computing on controls, projecting perturbed cells in
- UMAP visualization of perturbation-labeled cells

**Key Perturb-seq specific insight:** If you compute HVGs and PCA across all cells (including perturbed cells), genes whose expression is changed by strong perturbations will dominate the first principal components. This makes the embedding driven by perturbation identity rather than biological cell state. For exploratory QC and batch visualization, anchor to NTC cells.

**Estimated time:** 2 hours

In [ ]:
import os
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
from sklearn.decomposition import PCA

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_assigned.h5ad"))
print(adata)

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"

# Identify NTC cells
ntc_mask = adata.obs["is_ntc"] if "is_ntc" in adata.obs.columns else \
    adata.obs[PERT_COL].str.lower().str.contains("ctrl|control|ntc|non")
    
print(f"NTC cells: {ntc_mask.sum()}")

## 1. Normalization

Standard log-normalization:
1. Normalize total counts to 10,000 per cell (CPM-like but called "library size normalization")
2. Log1p transform: `log(x + 1)`

Raw counts must be preserved for pseudobulk DE analysis (notebook 08). Always store them in `layers['counts']` before normalizing `adata.X`.

In [ ]:
# Store raw counts before normalization
if "counts" not in adata.layers:
    import scipy.sparse
    X = adata.X
    if scipy.sparse.issparse(X):
        X_sample = X[:5, :5].toarray()
    else:
        X_sample = X[:5, :5]
    is_integer = np.allclose(X_sample, np.round(X_sample))
    
    if is_integer:
        print("adata.X contains raw counts — saving to layers['counts']")
        adata.layers["counts"] = adata.X.copy()
    elif adata.raw is not None:
        print("Using adata.raw for raw counts")
        raw_adata = adata.raw.to_adata()
        # Align genes
        common_genes = adata.var_names.intersection(raw_adata.var_names)
        adata.layers["counts"] = raw_adata[adata.obs_names, common_genes].X.copy()
    else:
        print("Warning: cannot find raw counts. Pseudobulk analysis in notebook 08 will require them.")
else:
    print("layers['counts'] already present.")

# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("Normalization complete. adata.X now contains log1p(CPM/100).")

## 2. NTC-anchored HVG selection

Select highly variable genes using **only NTC cells**. This ensures the HVG list reflects biological variation in unperturbed cells, not variation driven by the perturbations themselves.

In [ ]:
# Subset to NTC cells for HVG selection
ntc_adata = adata[ntc_mask].copy()
print(f"NTC subset for HVG selection: {ntc_adata.shape}")

sc.pp.highly_variable_genes(
    ntc_adata,
    n_top_genes=2000,
    flavor="seurat_v3",
    span=0.3,
)

hvg_mask = ntc_adata.var["highly_variable"]
hvg_genes = ntc_adata.var_names[hvg_mask]
print(f"HVGs selected from NTC cells: {len(hvg_genes)}")

# Transfer HVG annotation to the full AnnData
adata.var["highly_variable"] = adata.var_names.isin(hvg_genes)
adata.var["highly_variable"].value_counts()

## 3. Cell cycle scoring

Cell cycle stage is a major source of transcriptional variation in proliferating cells (like K562). Some perturbations alter cell cycle dynamics, which can confound DE analysis.

We score cells using the Tirosh et al. 2016 gene lists (included in scanpy).

In [ ]:
# Load cell cycle gene lists (S phase and G2M phase genes)
# scanpy includes these; they are based on Tirosh et al. 2016
s_genes = sc.datasets.builtins.get_cell_cycle_gene_lists()["s_genes"]
g2m_genes = sc.datasets.builtins.get_cell_cycle_gene_lists()["g2m_genes"]

In [ ]:
# Alternative: load from scanpy's built-in reference
# This approach works across scanpy versions
try:
    s_genes_ref = [
        "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG",
        "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "HELLS", "RFC2",
        "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7",
        "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1",
        "TIPIN", "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1",
        "CHAF1B", "BRIP1", "E2F8"
    ]
    g2m_genes_ref = [
        "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
        "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
        "TACC3", "PIMREG", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB",
        "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
        "CDCA3", "HMG20B", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
        "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR",
        "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF",
        "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
    ]

    # Filter to genes present in our dataset
    s_genes_present = [g for g in s_genes_ref if g in adata.var_names]
    g2m_genes_present = [g for g in g2m_genes_ref if g in adata.var_names]
    print(f"S phase genes present in dataset: {len(s_genes_present)}/{len(s_genes_ref)}")
    print(f"G2M phase genes present in dataset: {len(g2m_genes_present)}/{len(g2m_genes_ref)}")
    
    sc.tl.score_genes_cell_cycle(
        adata,
        s_genes=s_genes_present,
        g2m_genes=g2m_genes_present,
    )
    print("Cell cycle scoring complete.")
    print(adata.obs[["S_score", "G2M_score", "phase"]].head())

except Exception as e:
    print(f"Cell cycle scoring failed: {e}")
    print("Adding placeholder scores.")
    adata.obs["S_score"] = 0.0
    adata.obs["G2M_score"] = 0.0
    adata.obs["phase"] = "G1"

In [ ]:
# Visualize cell cycle phase distribution per perturbation
if "phase" in adata.obs.columns:
    phase_counts = adata.obs.groupby([PERT_COL, "phase"]).size().unstack(fill_value=0)
    phase_frac = phase_counts.div(phase_counts.sum(axis=1), axis=0)
    
    # Show top 20 perturbations sorted by S+G2M fraction
    sg2m_frac = phase_frac.get("S", 0) + phase_frac.get("G2M", 0)
    top_20 = sg2m_frac.nlargest(20).index
    
    fig, ax = plt.subplots(figsize=(12, 4))
    phase_frac.loc[top_20].plot(kind="bar", stacked=True, ax=ax,
                                 color={"G1": "#AED6F1", "S": "#F0B27A", "G2M": "#A9DFBF"})
    ax.set_xlabel("Perturbation (top 20 by S+G2M fraction)")
    ax.set_ylabel("Fraction of cells")
    ax.set_title("Cell cycle phase distribution per perturbation")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(title="Phase", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "07_cell_cycle_per_pert.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    print("Perturbations with high S+G2M fraction may be cell cycle activators.")
    print("Perturbations with high G1 fraction may be cell cycle inhibitors.")

## 4. NTC-anchored PCA

**Why anchor to NTC?**
- PCA fitted on all cells places strongly-perturbed cells far from the centroid, making the first PCs reflect perturbation identity
- Fitting PCA on NTC cells captures the "normal" cell state variation space
- Projecting perturbed cells into this NTC-derived space allows us to see *where* each perturbation moves cells relative to normal biology

In [ ]:
# Fit PCA on NTC cells
ntc_adata = adata[ntc_mask, adata.var["highly_variable"]].copy()

# Use sklearn PCA for explicit control over the fitting
from sklearn.preprocessing import StandardScaler

X_ntc = ntc_adata.X
if scipy.sparse.issparse(X_ntc):
    X_ntc = X_ntc.toarray()

# Scale features (zero mean, unit variance) — same scaling must be applied to perturbed cells
scaler = StandardScaler()
X_ntc_scaled = scaler.fit_transform(X_ntc)

pca = PCA(n_components=50, random_state=42)
X_ntc_pca = pca.fit_transform(X_ntc_scaled)

print(f"PCA fitted on {X_ntc.shape[0]} NTC cells, {X_ntc.shape[1]} HVGs")
print(f"Variance explained by first 5 PCs: {pca.explained_variance_ratio_[:5].round(3)}")

In [ ]:
# Project ALL cells into the NTC PCA space
X_all = adata[:, adata.var["highly_variable"]].X
if scipy.sparse.issparse(X_all):
    X_all = X_all.toarray()

X_all_scaled = scaler.transform(X_all)  # use NTC-fitted scaler
X_all_pca = pca.transform(X_all_scaled)  # use NTC-fitted PCA

adata.obsm["X_pca"] = X_all_pca
print(f"All cells projected into NTC PCA space: {X_all_pca.shape}")

In [ ]:
# Compare: standard PCA (all cells) vs. NTC-anchored PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard PCA on all cells
X_all_hvg = adata[:, adata.var["highly_variable"]].X
if scipy.sparse.issparse(X_all_hvg):
    X_all_hvg = X_all_hvg.toarray()
pca_all = PCA(n_components=2, random_state=42)
X_pca_standard = pca_all.fit_transform(StandardScaler().fit_transform(X_all_hvg))

# Color by NTC vs. perturbed
is_ntc_arr = ntc_mask.values
colors = ["#2196F3" if x else "#F44336" for x in is_ntc_arr]

for ax, coords, title in [
    (axes[0], X_pca_standard, "Standard PCA (all cells)"),
    (axes[1], X_all_pca[:, :2], "NTC-anchored PCA"),
]:
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.3, s=2, rasterized=True)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(title)

from matplotlib.patches import Patch
legend_elements = [
    Patch(color="#2196F3", label="NTC"),
    Patch(color="#F44336", label="Perturbed"),
]
axes[1].legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_pca_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

print("In NTC-anchored PCA, NTC cells cluster together and perturbed cells deviate from this.")

## 5. UMAP

In [ ]:
# Build neighbor graph on NTC-anchored PCA coordinates
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca")
sc.tl.umap(adata, min_dist=0.3, random_state=42)
print("UMAP computed.")

In [ ]:
# UMAP: color by perturbation type
adata.obs["pert_type"] = "perturbed"
adata.obs.loc[ntc_mask, "pert_type"] = "NTC"
if "is_dual" in adata.obs.columns:
    adata.obs.loc[adata.obs["is_dual"], "pert_type"] = "dual"

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Perturbation type
type_colors = {"NTC": "#2196F3", "perturbed": "#F44336", "dual": "#FF9800"}
umap = adata.obsm["X_umap"]
for pt_type, color in type_colors.items():
    mask = adata.obs["pert_type"] == pt_type
    axes[0].scatter(umap[mask, 0], umap[mask, 1], c=color, label=pt_type,
                    alpha=0.3, s=2, rasterized=True)
axes[0].set_title("Perturbation type")
axes[0].legend(markerscale=5)
axes[0].set_xlabel("UMAP 1")
axes[0].set_ylabel("UMAP 2")

# 2. Cell cycle phase
if "phase" in adata.obs.columns:
    phase_colors = {"G1": "#AED6F1", "S": "#F0B27A", "G2M": "#A9DFBF"}
    for phase, color in phase_colors.items():
        mask = adata.obs["phase"] == phase
        axes[1].scatter(umap[mask, 0], umap[mask, 1], c=color, label=phase,
                        alpha=0.3, s=2, rasterized=True)
    axes[1].set_title("Cell cycle phase")
    axes[1].legend(markerscale=5)
    axes[1].set_xlabel("UMAP 1")

# 3. Total UMI (QC sanity check)
if "total_counts" in adata.obs.columns:
    sc_plot = axes[2].scatter(umap[:, 0], umap[:, 1], c=adata.obs["total_counts"],
                              cmap="viridis", alpha=0.3, s=2, rasterized=True,
                              vmin=adata.obs["total_counts"].quantile(0.05),
                              vmax=adata.obs["total_counts"].quantile(0.95))
    plt.colorbar(sc_plot, ax=axes[2], label="Total UMI")
    axes[2].set_title("Total UMI count")
    axes[2].set_xlabel("UMAP 1")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_umap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Save processed AnnData

In [ ]:
out_path = os.path.join(DATA_DIR, "norman_processed.h5ad")
adata.write_h5ad(out_path)
print(f"Saved to {out_path}")
print(f"Shape: {adata.shape}")
print(f"Embeddings: {list(adata.obsm.keys())}")

## Key takeaways

1. Always store raw counts in `layers['counts']` before normalizing — pseudobulk DE (notebook 08) requires integer counts
2. **Compute HVGs and fit PCA using only NTC cells** — this prevents strong perturbation effects from dominating the embedding
3. Project all cells (including perturbed) into the NTC-derived PCA space for visualization
4. Cell cycle scoring is important in proliferating cell lines (K562) — some CRISPRa perturbations alter cell cycle, which can confound downstream analysis
5. The UMAP is for visual QC and hypothesis generation — not for quantitative perturbation effect analysis (use pseudobulk DE for that)

**Next (parallel):**
- [08_perturbation_effects.ipynb](08_perturbation_effects.ipynb) — uses Replogle 2022 dataset
- [09_mixscape.ipynb](09_mixscape.ipynb) — uses Norman 2019 (this file)